# Week 5: Support Vector Machines, Kernel Trick, and Regularization

## 1. Setup Notebook
- Import libraries  
- Define functions for evaluating SVM models

In [1]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

#evaluate SVM model
def evaluate_svm(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return {"Accuracy": acc, "Confusion Matrix": cm, "Report": report}

## 2. Dataset 1: customer_churn
### 2.1 Data Overview & Preparation

In [2]:
#load data
train_cc = pd.read_csv('customer_churn_dataset-training-master.csv')
test_cc = pd.read_csv('customer_churn_dataset-testing-master.csv')
#combine train and test set
customer_churn = pd.concat([train_cc, test_cc], ignore_index=True)
#drop null rows
customer_churn.dropna(inplace=True)
#take sample of dataset for quicker run time
sample, _ = train_test_split(
    customer_churn,
    train_size = 0.3, #use 30% of dataset
    stratify = customer_churn['Churn'],
    random_state = 42
)
X = pd.get_dummies(sample.drop("Churn", axis=1, errors="ignore"), drop_first=True)
y = sample["Churn"] if "Churn" in sample.columns else None

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

### 2.2 Linear SVM

In [3]:
if y is not None:
    svm_linear = SVC(kernel="linear", C=1)
    results_linear = evaluate_svm(svm_linear, X_train, X_test, y_train, y_test)
    print("Linear SVM:", results_linear)

Linear SVM: {'Accuracy': 0.9023191370039257, 'Confusion Matrix': array([[11260,  2177],
       [  784, 16092]]), 'Report': '              precision    recall  f1-score   support\n\n         0.0       0.93      0.84      0.88     13437\n         1.0       0.88      0.95      0.92     16876\n\n    accuracy                           0.90     30313\n   macro avg       0.91      0.90      0.90     30313\nweighted avg       0.90      0.90      0.90     30313\n'}


### 2.3 Kernel SVMs (RBF, Polynomial, Sigmoid)

In [4]:
if y is not None:
    kernels = ["rbf", "poly", "sigmoid"]
    for k in kernels:
        svm = SVC(kernel=k, C=1)
        results = evaluate_svm(svm, X_train, X_test, y_train, y_test)
        print(f"SVM with {k} kernel:", results)

SVM with rbf kernel: {'Accuracy': 0.9322402929436215, 'Confusion Matrix': array([[11621,  1816],
       [  238, 16638]]), 'Report': '              precision    recall  f1-score   support\n\n         0.0       0.98      0.86      0.92     13437\n         1.0       0.90      0.99      0.94     16876\n\n    accuracy                           0.93     30313\n   macro avg       0.94      0.93      0.93     30313\nweighted avg       0.94      0.93      0.93     30313\n'}
SVM with poly kernel: {'Accuracy': 0.9243558869132056, 'Confusion Matrix': array([[11454,  1983],
       [  310, 16566]]), 'Report': '              precision    recall  f1-score   support\n\n         0.0       0.97      0.85      0.91     13437\n         1.0       0.89      0.98      0.94     16876\n\n    accuracy                           0.92     30313\n   macro avg       0.93      0.92      0.92     30313\nweighted avg       0.93      0.92      0.92     30313\n'}
SVM with sigmoid kernel: {'Accuracy': 0.7995579454359516, '

### 2.4 Regularization (Effect of C parameter)

In [ ]:
if y is not None:
    for c in [0.1, 1, 10]:
        svm = SVC(kernel="linear", C=c)
        results = evaluate_svm(svm, X_train, X_test, y_train, y_test)
        print(f"Linear SVM with C={c}:", results)

Linear SVM with C=0.1: {'Accuracy': 0.9023191370039257, 'Confusion Matrix': array([[11260,  2177],
       [  784, 16092]]), 'Report': '              precision    recall  f1-score   support\n\n         0.0       0.93      0.84      0.88     13437\n         1.0       0.88      0.95      0.92     16876\n\n    accuracy                           0.90     30313\n   macro avg       0.91      0.90      0.90     30313\nweighted avg       0.90      0.90      0.90     30313\n'}
Linear SVM with C=1: {'Accuracy': 0.9023191370039257, 'Confusion Matrix': array([[11260,  2177],
       [  784, 16092]]), 'Report': '              precision    recall  f1-score   support\n\n         0.0       0.93      0.84      0.88     13437\n         1.0       0.88      0.95      0.92     16876\n\n    accuracy                           0.90     30313\n   macro avg       0.91      0.90      0.90     30313\nweighted avg       0.90      0.90      0.90     30313\n'}


#### Relevant plots
Decision boundary for 2 key features:

In [ ]:
features = ["Tenure", "Total Spend"]
X_plot = customer_churn[features]
y_plot = customer_churn["Churn"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_plot)
svm = SVC(kernel="rbf", C=1.0, gamma=0.1)
svm.fit(X_scaled, y_plot)

#meshgrid for decision region
xx, yy = np.meshgrid(
    np.linspace(X_scaled[:,0].min(), X_scaled[:,0].max(), 200),
    np.linspace(X_scaled[:,1].min(), X_scaled[:,1].max(), 200)
)
Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contourf(xx, yy, Z, cmap="coolwarm", alpha=0.3)
plt.scatter(X_scaled[:,0], X_scaled[:,1], c=y_plot, cmap="coolwarm", s=30)
plt.title("SVM Decision Boundary (RBF Kernel)")
plt.xlabel(features[0])
plt.ylabel(features[1])
plt.show()

## 3. Dataset 2: digital_marketing_campaign
### 3.1 Data Overview & Preparation

In [ ]:
digital_marketing = pd.read_csv("digital_marketing_campaign_dataset.csv")
digital_marketing.dropna(inplace=True)
X = pd.get_dummies(digital_marketing.drop("Conversion", axis=1, errors="ignore"), drop_first=True)
y = digital_marketing["Conversion"] if "Conversion" in digital_marketing.columns else None

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

### 3.2 Linear SVM

In [ ]:
if y is not None:
    svm_linear = SVC(kernel="linear", C=1)
    results_linear = evaluate_svm(svm_linear, X_train, X_test, y_train, y_test)
    print("Linear SVM:", results_linear)

Linear SVM: {'Accuracy': 0.87875, 'Confusion Matrix': array([[   0,  194],
       [   0, 1406]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.00      0.00      0.00       194\n           1       0.88      1.00      0.94      1406\n\n    accuracy                           0.88      1600\n   macro avg       0.44      0.50      0.47      1600\nweighted avg       0.77      0.88      0.82      1600\n'}


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### 3.3 Kernel SVMs (RBF, Polynomial, Sigmoid)

In [ ]:
if y is not None:
    kernels = ["rbf", "poly", "sigmoid"]
    for k in kernels:
        svm = SVC(kernel=k, C=1)
        results = evaluate_svm(svm, X_train, X_test, y_train, y_test)
        print(f"SVM with {k} kernel:", results)

SVM with rbf kernel: {'Accuracy': 0.895625, 'Confusion Matrix': array([[  55,  139],
       [  28, 1378]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.66      0.28      0.40       194\n           1       0.91      0.98      0.94      1406\n\n    accuracy                           0.90      1600\n   macro avg       0.79      0.63      0.67      1600\nweighted avg       0.88      0.90      0.88      1600\n'}
SVM with poly kernel: {'Accuracy': 0.885625, 'Confusion Matrix': array([[  46,  148],
       [  35, 1371]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.57      0.24      0.33       194\n           1       0.90      0.98      0.94      1406\n\n    accuracy                           0.89      1600\n   macro avg       0.74      0.61      0.64      1600\nweighted avg       0.86      0.89      0.86      1600\n'}
SVM with sigmoid kernel: {'Accuracy': 0.875625, 'Confusion Matrix': array([[  29,  165]

### 3.4 Regularization (Effect of C parameter)

In [ ]:
if y is not None:
    for c in [0.1, 1, 10]:
        svm = SVC(kernel="linear", C=c)
        results = evaluate_svm(svm, X_train, X_test, y_train, y_test)
        print(f"Linear SVM with C={c}:", results)

/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Linear SVM with C=0.1: {'Accuracy': 0.87875, 'Confusion Matrix': array([[   0,  194],
       [   0, 1406]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.00      0.00      0.00       194\n           1       0.88      1.00      0.94      1406\n\n    accuracy                           0.88      1600\n   macro avg       0.44      0.50      0.47      1600\nweighted avg       0.77      0.88      0.82      1600\n'}


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Linear SVM with C=1: {'Accuracy': 0.87875, 'Confusion Matrix': array([[   0,  194],
       [   0, 1406]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.00      0.00      0.00       194\n           1       0.88      1.00      0.94      1406\n\n    accuracy                           0.88      1600\n   macro avg       0.44      0.50      0.47      1600\nweighted avg       0.77      0.88      0.82      1600\n'}
Linear SVM with C=10: {'Accuracy': 0.87875, 'Confusion Matrix': array([[   0,  194],
       [   0, 1406]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.00      0.00      0.00       194\n           1       0.88      1.00      0.94      1406\n\n    accuracy                           0.88      1600\n   macro avg       0.44      0.50      0.47      1600\nweighted avg       0.77      0.88      0.82      1600\n'}


/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/codespace/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 4. Dataset 3: marketing_campaign (Optional for Week 5)
### 4.1 Data Overview & Preparation

In [ ]:
marketing_campaign = pd.read_csv("marketing_campaign.csv", sep=';')
marketing_campaign.dropna(inplace=True)
X = pd.get_dummies(marketing_campaign.drop("Response", axis=1, errors="ignore"), drop_first=True)
y = marketing_campaign["Response"] if "Response" in marketing_campaign.columns else None

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

### 4.2 Linear SVM

In [ ]:
if y is not None:
    svm_linear = SVC(kernel="linear", C=1)
    results_linear = evaluate_svm(svm_linear, X_train, X_test, y_train, y_test)
    print("Linear SVM:", results_linear)

Linear SVM: {'Accuracy': 0.831081081081081, 'Confusion Matrix': array([[326,  56],
       [ 19,  43]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.94      0.85      0.90       382\n           1       0.43      0.69      0.53        62\n\n    accuracy                           0.83       444\n   macro avg       0.69      0.77      0.72       444\nweighted avg       0.87      0.83      0.85       444\n'}


### 4.3 Kernel SVMs (RBF, Polynomial, Sigmoid)

In [ ]:
if y is not None:
    kernels = ["rbf", "poly", "sigmoid"]
    for k in kernels:
        svm = SVC(kernel=k, C=1)
        results = evaluate_svm(svm, X_train, X_test, y_train, y_test)
    print(f"SVM with {k} kernel:", results)

SVM with sigmoid kernel: {'Accuracy': 0.8536036036036037, 'Confusion Matrix': array([[368,  14],
       [ 51,  11]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.88      0.96      0.92       382\n           1       0.44      0.18      0.25        62\n\n    accuracy                           0.85       444\n   macro avg       0.66      0.57      0.59       444\nweighted avg       0.82      0.85      0.83       444\n'}


### 4.4 Regularization (Effect of C parameter)

In [ ]:
if y is not None:
    for c in [0.1, 1, 10]:
        svm = SVC(kernel="linear", C=c)
        results = evaluate_svm(svm, X_train, X_test, y_train, y_test)
        print(f"Linear SVM with C={c}:", results)

Linear SVM with C=0.1: {'Accuracy': 0.8761261261261262, 'Confusion Matrix': array([[354,  28],
       [ 27,  35]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.93      0.93      0.93       382\n           1       0.56      0.56      0.56        62\n\n    accuracy                           0.88       444\n   macro avg       0.74      0.75      0.74       444\nweighted avg       0.88      0.88      0.88       444\n'}
Linear SVM with C=1: {'Accuracy': 0.831081081081081, 'Confusion Matrix': array([[326,  56],
       [ 19,  43]]), 'Report': '              precision    recall  f1-score   support\n\n           0       0.94      0.85      0.90       382\n           1       0.43      0.69      0.53        62\n\n    accuracy                           0.83       444\n   macro avg       0.69      0.77      0.72       444\nweighted avg       0.87      0.83      0.85       444\n'}
Linear SVM with C=10: {'Accuracy': 0.7184684684684685, 'Confusion Matrix': 